# Safety Jacket Detection - Training with Roboflow Dataset (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anzonathan/Traffic-Police-Agent/blob/main/notebooks/train_jacket_model_colab.ipynb)

> ⚠️ **Before running:** Set runtime to GPU — `Runtime → Change runtime type → T4 GPU`

> 🔑 **Add your Roboflow API key** in the Secrets sidebar (🔑 icon) as `ROBOFLOW_API_KEY`

**Dataset:** Manually annotated on Roboflow (`score-board-pro/traffiic-agent`)

**Pipeline:**
1. Install dependencies
2. Download annotated dataset from Roboflow
3. Prepare dataset (fix paths, create val split if needed)
4. Train **YOLOv8**
5. Download `best.pt` to your local machine

## Step 1: Verify GPU & Install Dependencies

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU ready: {torch.cuda.get_device_name(0)}")

In [ ]:
!pip install -q ultralytics roboflow
print("Dependencies installed!")

## Step 2: Download Annotated Dataset from Roboflow

> 🔑 Make sure you added `ROBOFLOW_API_KEY` in the Secrets sidebar (🔑 icon on the left)

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("score-board-pro").project("traffiic-agent")
version = project.version(1)
dataset = version.download("yolov8")

print(f"\nDataset downloaded to: {dataset.location}")

## Step 3: Prepare Dataset

Fixes paths in `data.yaml` and automatically creates an 80/20 train/val split if Roboflow didn't include a validation set.

In [ ]:
import os, yaml, shutil, random

dataset_root = os.path.abspath(dataset.location)
data_yaml_path = os.path.join(dataset_root, "data.yaml")

train_img_dir = os.path.join(dataset_root, "train", "images")
train_lbl_dir = os.path.join(dataset_root, "train", "labels")
valid_img_dir = os.path.join(dataset_root, "valid", "images")
valid_lbl_dir = os.path.join(dataset_root, "valid", "labels")

# Check if valid split exists and has images
has_valid = os.path.exists(valid_img_dir) and len(os.listdir(valid_img_dir)) > 0

if not has_valid:
    print("⚠️  No validation split found. Creating 80/20 split from training data...")
    os.makedirs(valid_img_dir, exist_ok=True)
    os.makedirs(valid_lbl_dir, exist_ok=True)

    # Get all training images
    all_images = sorted([f for f in os.listdir(train_img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
    random.seed(42)
    random.shuffle(all_images)

    # Move 20% to validation
    val_count = max(1, int(len(all_images) * 0.2))
    val_images = all_images[:val_count]

    for img_name in val_images:
        # Move image
        shutil.move(os.path.join(train_img_dir, img_name), os.path.join(valid_img_dir, img_name))
        # Move corresponding label
        lbl_name = os.path.splitext(img_name)[0] + ".txt"
        lbl_src = os.path.join(train_lbl_dir, lbl_name)
        if os.path.exists(lbl_src):
            shutil.move(lbl_src, os.path.join(valid_lbl_dir, lbl_name))

    print(f"   Moved {val_count} images to validation.")
else:
    print("✅ Validation split already exists.")

# Fix data.yaml with absolute paths
with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = dataset_root
data_cfg['train'] = 'train/images'
data_cfg['val'] = 'valid/images'
if 'test' in data_cfg:
    del data_cfg['test']  # remove test if no test set

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

# Verify
train_count = len(os.listdir(train_img_dir))
valid_count = len(os.listdir(valid_img_dir))
print(f"\nFinal dataset:")
print(f"  train: ✅ {train_count} images")
print(f"  valid: ✅ {valid_count} images")
print(f"\ndata.yaml:")
with open(data_yaml_path) as f:
    print(f.read())

## Step 4: Train YOLOv8

> ⏳ Training may take 15-40 min on a T4 GPU depending on dataset size.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    device="cuda",
    patience=20,
    augment=True,
    fliplr=0.5,
    mosaic=1.0,
    project="runs/detect",
    name="safety_jacket_v2"
)
print("Training complete!")

## Step 5: Evaluate & Download `best.pt`

After downloading, place it in your local project at `models/best_jacket.pt`.

In [ ]:
import os, glob
from google.colab import files

best_model_path = "runs/detect/safety_jacket_v2/weights/best.pt"

# If the exact path doesn't exist, search for it
if not os.path.exists(best_model_path):
    candidates = glob.glob("runs/detect/*/weights/best.pt")
    if candidates:
        best_model_path = candidates[-1]
        print(f"Found model at: {best_model_path}")
    else:
        print("ERROR: No best.pt found! Check runs/ directory:")
        !ls -R runs/

if os.path.exists(best_model_path):
    trained_model = YOLO(best_model_path)
    metrics = trained_model.val(data=data_yaml_path)
    print(f"\nmAP50:    {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")
    
    print("\nDownloading best.pt...")
    files.download(best_model_path)
    print("Done! Move the downloaded file to: models/best_jacket.pt")